
Cluster review pipeline for WEME individual id.
1. Remove clusters with <5 detections
2. ARU group check - which ARUs hear each cluster the most?
3. Temporal overlap check - which clusters sing at the same time?
4. Generate merge candidate spreadsheet for review


In [ ]:

import pandas as pd
import numpy as np
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from pathlib import Path
import os

# ------------- CONFIGURATION -------------------------
RESULTS_CSV = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/individual_id/individual_id_results.csv'
OUTPUT_DIR  = '/media/UofA/BU_Work/BayneLabWorkSpace/Katrine_workspace/Localization_per_day/OKLG-8-20250615/individual_id/cluster_review'
MIN_SONGS   = 5      # minimum detections to keep a cluster
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------- 1. Load data and remove clusters -----------------------------
df = pd.read_csv(RESULTS_CSV)
df['timestamp_pdt'] = pd.to_datetime(df['timestamp_pdt'])
df['receiver_ids']  = df['receiver_ids'].apply(json.loads)

# Work with HDBSCAN mcs=7 only
df = df[df['cluster_hdbscan_7'] != -1].copy()  # drop noise
df['cluster'] = df['cluster_hdbscan_7'].astype(int)

print(f"\nremove clusters with <5 detections")

cluster_counts = df['cluster'].value_counts().sort_index()
print("\nDetections per cluster:")
for cid, n in cluster_counts.items():
    status = '✅ keep' if n >= MIN_SONGS else '❌ remove'
    print(f"  Cluster {cid:>2}: n={n:>3}  {status}")

keep_clusters = cluster_counts[cluster_counts >= MIN_SONGS].index.tolist()
remove_clusters = cluster_counts[cluster_counts < MIN_SONGS].index.tolist()
df = df[df['cluster'].isin(keep_clusters)].copy()

print(f"\n  Kept:    {len(keep_clusters)} clusters")
print(f"  Removed: {len(remove_clusters)} clusters {remove_clusters}")

# -------------- 2. ARU group check ---------------------
# For each cluster, count how many times each ARU appeared in detections
all_arus = sorted(set(aru for ids in df['receiver_ids'] for aru in ids))

aru_records = []
for cid in sorted(df['cluster'].unique()):
    sub  = df[df['cluster'] == cid]
    n    = len(sub)
    # Count ARU frequency
    aru_counts = {aru: 0 for aru in all_arus}
    for ids in sub['receiver_ids']:
        for aru in ids:
            if aru in aru_counts:
                aru_counts[aru] += 1
    # Top ARUs (heard in > 30% of detections)
    top_arus = sorted([a for a, c in aru_counts.items() if c / n > 0.3],
                      key=lambda a: -aru_counts[a])
    aru_records.append({
        'cluster':      cid,
        'n_detections': n,
        'top_arus':     ', '.join(top_arus),
        'centroid_x':   round(sub['x'].mean(), 1),
        'centroid_y':   round(sub['y'].mean(), 1),
        'spatial_spread_m': round(
            np.sqrt(sub['x'].var() + sub['y'].var()), 1
        ),
        **{f'aru_{a}': round(aru_counts[a] / n * 100, 0) for a in all_arus}
    })

df_aru = pd.DataFrame(aru_records)
print("\nTop ARUs per cluster (heard in >30% of detections):")
print(df_aru[['cluster', 'n_detections', 'top_arus',
              'centroid_x', 'centroid_y', 'spatial_spread_m']].to_string(index=False))

# ---------------- 3. temporal overlap check ---------------------

# For each pair of clusters, check if any detections overlap in time
# Using a 3 second window size
WINDOW_SEC = 3

overlap_records = []
cluster_ids = sorted(df['cluster'].unique())

for c1, c2 in combinations(cluster_ids, 2):
    t1 = df[df['cluster'] == c1]['timestamp_pdt'].sort_values().values
    t2 = df[df['cluster'] == c2]['timestamp_pdt'].sort_values().values

    # Check for any overlapping 3-second windows
    overlaps = 0
    for time1 in t1:
        for time2 in t2:
            diff = abs((pd.Timestamp(time1) - pd.Timestamp(time2)).total_seconds())
            if diff <= WINDOW_SEC:
                overlaps += 1

    # Time ranges
    c1_range = (df[df['cluster'] == c1]['timestamp_pdt'].min(),
                df[df['cluster'] == c1]['timestamp_pdt'].max())
    c2_range = (df[df['cluster'] == c2]['timestamp_pdt'].min(),
                df[df['cluster'] == c2]['timestamp_pdt'].max())

    # Do time ranges overlap at all?
    ranges_overlap = (c1_range[0] <= c2_range[1]) and (c2_range[0] <= c1_range[1])

    # Spatial distance between centroids
    x1, y1 = df[df['cluster'] == c1][['x', 'y']].mean()
    x2, y2 = df[df['cluster'] == c2][['x', 'y']].mean()
    centroid_dist = round(np.sqrt((x1 - x2)**2 + (y1 - y2)**2), 1)

    # Same ARU group?
    top1 = set(df_aru[df_aru['cluster'] == c1]['top_arus'].values[0].split(', '))
    top2 = set(df_aru[df_aru['cluster'] == c2]['top_arus'].values[0].split(', '))
    aru_overlap = len(top1 & top2)
    aru_similarity = round(aru_overlap / max(len(top1 | top2), 1) * 100, 0)

    # Merge logic
    simultaneous    = overlaps > 0          # singing at same time → different birds
    never_overlap   = not ranges_overlap    # never active together → merge candidate
    similar_arus    = aru_similarity >= 50  # same ARU group → same territory
    merge_candidate = (not simultaneous) and similar_arus

    overlap_records.append({
        'cluster_A':        c1,
        'cluster_B':        c2,
        'simultaneous_events': overlaps,
        'time_ranges_overlap': ranges_overlap,
        'centroid_dist_m':  centroid_dist,
        'aru_similarity_%': aru_similarity,
        'shared_arus':      ', '.join(top1 & top2),
        'merge_candidate':  merge_candidate,
        'reason': (
            'Same ARU group, never simultaneous' if merge_candidate
            else ('Simultaneous singing — different birds' if simultaneous
            else 'Different ARU groups — different territories')
        )
    })

df_overlap = pd.DataFrame(overlap_records)

print("\nMerge candidates (same ARU group + no simultaneous singing):")
merge_df = df_overlap[df_overlap['merge_candidate']].sort_values('aru_similarity_%', ascending=False)
if len(merge_df) == 0:
    print("  None found — all clusters appear to be distinct individuals")
else:
    print(merge_df[['cluster_A', 'cluster_B', 'simultaneous_events',
                     'centroid_dist_m', 'aru_similarity_%',
                     'shared_arus', 'reason']].to_string(index=False))

print("\nDefinitely different birds (simultaneous singing):")
diff_df = df_overlap[df_overlap['simultaneous_events'] > 0].sort_values(
    'simultaneous_events', ascending=False)
print(diff_df[['cluster_A', 'cluster_B', 'simultaneous_events',
               'centroid_dist_m', 'aru_similarity_%']].to_string(index=False))

# ----------------------- 4. generate review spreadsheet ----------------------
# Main cluster summary sheet
df_summary = df_aru.copy()
df_summary['time_start']    = df.groupby('cluster')['timestamp_pdt'].min().values
df_summary['time_end']      = df.groupby('cluster')['timestamp_pdt'].max().values
df_summary['duration_hrs']  = (
    (pd.to_datetime(df_summary['time_end']) -
     pd.to_datetime(df_summary['time_start'])).dt.total_seconds() / 3600
).round(2)

# Add merge candidate info
merge_candidates_per_cluster = {}
for _, row in merge_df.iterrows():
    for c in [row['cluster_A'], row['cluster_B']]:
        other = row['cluster_B'] if c == row['cluster_A'] else row['cluster_A']
        if c not in merge_candidates_per_cluster:
            merge_candidates_per_cluster[c] = []
        merge_candidates_per_cluster[c].append(str(other))

df_summary['merge_candidate_with'] = df_summary['cluster'].map(
    lambda c: ', '.join(merge_candidates_per_cluster.get(c, []))
)
df_summary['annotation_song_variant'] = ''  # blank for manual entry
df_summary['annotation_keep_merge_remove'] = ''  # blank for manual entry
df_summary['annotation_notes'] = ''  # blank for manual entry

# Per-detection sheet (for clip-by-clip review)
df_clips = df[[
    'cluster', 'timestamp_pdt', 'x', 'y', 'residual_rms',
    'mean_cc_max', 'n_receivers', 'receiver_ids', 'clip_path'
]].copy().sort_values(['cluster', 'timestamp_pdt'])
df_clips['annotation_song_variant'] = ''
df_clips['annotation_is_weme'] = ''
df_clips['annotation_notes'] = ''

# Save to Excel with multiple sheets
excel_path = os.path.join(OUTPUT_DIR, 'cluster_review.xlsx')
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_summary.to_excel(writer, sheet_name='Cluster_Summary', index=False)
    df_clips.to_excel(writer, sheet_name='Per_Detection', index=False)
    df_overlap.to_excel(writer, sheet_name='Pairwise_Overlap', index=False)
    merge_df.to_excel(writer, sheet_name='Merge_Candidates', index=False)

print(f"  Spreadsheet saved to: {excel_path}")
print(f"  Sheets: Cluster_Summary | Per_Detection | Pairwise_Overlap | Merge_Candidates")

# -------------------- 5. aru heatmap plot --------------------------
aru_cols = [c for c in df_aru.columns if c.startswith('aru_')]
aru_matrix = df_aru.set_index('cluster')[aru_cols].copy()
aru_matrix.columns = [c.replace('aru_', '') for c in aru_matrix.columns]

fig, ax = plt.subplots(figsize=(max(12, len(all_arus)), 6))
sns.heatmap(aru_matrix, annot=True, fmt='.0f', cmap='YlOrRd',
            ax=ax, cbar_kws={'label': '% detections heard by this ARU'})
ax.set_xlabel('ARU ID')
ax.set_ylabel('Cluster')
ax.set_title('ARU Hearing Frequency per Cluster (%)\nClusters with similar ARU patterns = merge candidates')
plt.tight_layout()
heatmap_path = os.path.join(OUTPUT_DIR, 'aru_heatmap.png')
plt.savefig(heatmap_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"  ARU heatmap saved to: {heatmap_path}")

# ------------------- summary --------------------------------
print(f"  Clusters after step 1 (≥{MIN_SONGS} detections): {len(keep_clusters)}")
print(f"  Merge candidates identified:                     {len(merge_df)}")
print(f"  Confirmed different birds (simultaneous):        {len(diff_df)}")
print(f"\n  Next steps:")
print(f"  1. Open cluster_review.xlsx")
print(f"  2. Listen to clips in Per_Detection sheet")
print(f"  3. Fill in 'annotation_song_variant' column")
print(f"  4. Check Merge_Candidates sheet and fill in 'annotation_keep_merge_remove'")
print(f"  5. Use aru_heatmap.png to visually confirm ARU group similarities")

